In [ ]:
import sys, os

# ── Fijar working directory a la raíz del proyecto ──────────────────
# Solo subir si todavía estamos dentro de notebooks/
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

print(f"📂 Raíz del proyecto: {os.getcwd()}")
sys.path.insert(0, os.getcwd())  # asegura que src/ sea importable

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from src.data_utils import load_config, load_raw_data, get_missing_summary
from src.visualization import plot_target_distribution, plot_correlation_heatmap

# ── Cargar config y datos ────────────────────────────────────────────
config = load_config("config/config.yaml")   # ← sin "..", ya estamos en la raíz
df = load_raw_data(config)
df.head(3)

FileNotFoundError: [Errno 2] No such file or directory: '../config/config.yaml'

In [ ]:
print("=== FORMA DEL DATASET ===")
print(f"Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}")
print("\n=== TIPOS DE DATOS ===")
print(df.dtypes.value_counts())
print("\n=== PRIMERAS FILAS ===")
df.head(3)

In [ ]:

df[config["features_numericas"]].describe().round(3)

In [ ]:
missing = get_missing_summary(df)
print("=== VALORES FALTANTES ===")
print(missing if len(missing) > 0 else "✅ No hay valores faltantes significativos")

In [ ]:

plot_target_distribution(df, "Stream", log=True)

In [ ]:

plot_target_distribution(df, "Views", log=True)

In [ ]:

all_feats = config["features_numericas"] + ["Stream"]
plot_correlation_heatmap(df, [f for f in all_feats if f in df.columns])

In [ ]:
top_artists = (df.groupby("Artist")["Stream"]
               .sum()
               .sort_values(ascending=False)
               .head(10) / 1e9)
fig, ax = plt.subplots(figsize=(10, 5))
top_artists.plot(kind="barh", ax=ax, color="#1E88E5")
ax.set_xlabel("Streams totales (miles de millones)")
ax.set_title("Top 10 Artistas por Streams en Spotify")
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
df["Album_type"].value_counts().plot(kind="bar", ax=ax, color=["#42A5F5","#EF5350","#66BB6A"])
ax.set_title("Distribución por Tipo de Álbum")
ax.set_xlabel("Tipo")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("../reports/figures/album_type_dist.png", dpi=150)
plt.show()

In [ ]:
key_feats = ["Danceability","Energy","Valence","Loudness","Tempo","Stream"]
sample = df[key_feats].dropna().sample(min(500, len(df)), random_state=42)
sample["log_Stream"] = np.log1p(sample["Stream"])
sns.pairplot(sample[["Danceability","Energy","Valence","Loudness","log_Stream"]],
             diag_kind="kde", plot_kws={"alpha":0.3, "s":10})
plt.savefig("../reports/figures/pairplot_features.png", dpi=150)
plt.show()